To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [27]:
# Install required packages
!uv pip install jsonlines openpyxl pandas tabulate seaborn matplotlib tenacity tqdm rank-bm25 sentence-transformers

# Optional: install flash-attn for faster inference (may require CUDA)
# !pip install flash-attn --no-build-isolation

Using Python 3.12.13 environment at: /usr
Resolved 72 packages in 301ms
Prepared 2 packages in 7ms
Installed 2 packages in 2ms
 + jsonlines==4.0.0
 + rank-bm25==0.2.2


In [3]:
#@title Colab Extra Install
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [29]:
import os
import sys
from IPython import get_ipython


def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"Data Path: {base_data_path}")
    print(f"Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name


def load_secret(key_name: str) -> str | None:
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata

            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient

            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch

        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi


INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")

Environment: Google Colab
Data Path: /content/
Output Path: /content/

🔧 System Information
Python version: 3.12.13
PyTorch version: 2.7.0+cu126
CUDA version: 12.6
GPU count: 1
  GPU 0: Tesla T4
Sat Jun 20 08:47:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /  

In [30]:
%cd {OUTPUT_PATH}
!rm -rf ToolFormer
!git clone https://{GITHUB_TOKEN}@github.com/dzungphieuluuky/ToolFormer.git
%cd {OUTPUT_PATH}/ToolFormer

/content
Cloning into 'ToolFormer'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (232/232), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 232 (delta 113), reused 191 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (232/232), 1.49 MiB | 6.89 MiB/s, done.
Resolving deltas: 100% (113/113), done.
/content/ToolFormer


In [31]:
!git branch

* main


In [32]:
import os
import sys
import json
import re
import math
import random
import time
import uuid
import warnings
import logging
from pathlib import Path
from typing import Any, Optional, List, Dict, Callable, Tuple
from dataclasses import dataclass, field, asdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from tabulate import tabulate

# Transformers / TRL / Unsloth
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel, PatchFastRL
from vllm import SamplingParams

# Datasets
try:
    from datasets import Dataset
except ImportError:
    Dataset = None

# JSON lines
try:
    import jsonlines
except ImportError:
    jsonlines = None

# Tenacity for retries
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type, before_sleep_log

# Rich logging
from rich.logging import RichHandler

# Set warnings
warnings.filterwarnings("ignore")

# Set random seeds
SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("All imports successful.")

RuntimeError: Failed to import trl.trainer.grpo_trainer because of the following error (look up to see its traceback):
cannot import name 'is_rich_available' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

In [ ]:
# ===================== logging_utils.py =====================
def get_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = RichHandler(rich_tracebacks=True, show_time=True, markup=True)
        logger.addHandler(handler)
        logger.setLevel(level)
        logger.propagate = False
    return logger

# ===================== model_utils.py =====================
def save_model_and_tokenizer(model, tokenizer, output_dir: str) -> None:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"[model_utils] Saved → {output_dir}")

def load_model_from_path(
    model_path: str,
    base_model_name: str = "unsloth/Qwen3-4B-Base",
    max_seq_length: int = 2048,
    load_in_4bit: bool = True,
):
    """Load a fine-tuned LoRA adapter on top of the base model."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model_name,
        max_seq_length=max_seq_length,
        load_in_4bit=load_in_4bit,
        dtype=None,
    )
    model.load_adapter(model_path)
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def generate_response(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 512,
    temperature: float = 0.6,
    do_sample: bool = True,
) -> str:
    """Single-sample inference helper."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)

In [ ]:
# ===================== sandbox.py =====================
def _default_mock(function_name: str, arguments: dict) -> dict:
    return {
        "status": "success",
        "function": function_name,
        "result": f"Mock result for {function_name}({arguments})",
    }

class Sandbox:
    def __init__(
        self,
        function_library: dict,
        mocks: dict[str, Callable] | None = None,
        timeout_seconds: float = 5.0,
    ):
        self.library = function_library
        self.mocks = mocks or {}
        self.timeout = timeout_seconds
        self._call_log: list[dict] = []

    def execute(self, call_input: str | dict | None) -> bool:
        if call_input is None:
            return False
        call = self._resolve_call(call_input)
        if call is None:
            return False
        return self._run_call(call)

    def execute_all(self, response: str) -> list[bool]:
        from src.reward.base_reward import extract_all_calls
        calls = extract_all_calls(response)
        return [self._run_call(c) for c in calls]

    def get_call_log(self) -> list[dict]:
        return self._call_log.copy()

    def clear_log(self) -> None:
        self._call_log.clear()

    def _resolve_call(self, call_input: str | dict) -> dict | None:
        if isinstance(call_input, dict):
            return call_input
        from src.reward.base_reward import extract_call
        return extract_call(call_input)

    def _run_call(self, call: dict) -> bool:
        func_name = call.get("function")
        arguments = call.get("arguments", {})
        if not func_name:
            return False
        if func_name not in self.library:
            self._log(func_name, arguments, "error", "function_not_found")
            return False
        schema = self.library[func_name]
        if not self._validate_args(func_name, arguments, schema):
            return False
        try:
            mock_fn = self.mocks.get(func_name, _default_mock)
            result = mock_fn(**arguments) if mock_fn != _default_mock else _default_mock(func_name, arguments)
            self._log(func_name, arguments, "success", result)
            return True
        except Exception as exc:
            self._log(func_name, arguments, "error", str(exc))
            return False

    def _validate_args(self, func_name: str, arguments: dict, schema: dict) -> bool:
        params = schema.get("parameters", {})
        constraints = schema.get("constraints", {})
        for pname, pinfo in params.items():
            if pinfo.get("required", False) and pname not in arguments:
                self._log(func_name, arguments, "error", f"missing_required:{pname}")
                return False
        for pname, con in constraints.items():
            if pname not in arguments:
                continue
            val = arguments[pname]
            if "min" in con and val < con["min"]:
                return False
            if "max" in con and val > con["max"]:
                return False
            if "enum" in con and val not in con["enum"]:
                return False
        return True

    def _log(self, func_name: str, arguments: dict, status: str, result: Any) -> None:
        self._call_log.append({
            "function": func_name,
            "arguments": arguments,
            "status": status,
            "result": result,
        })

In [ ]:
# ===================== retrieval.py (data structures) =====================
@dataclass
class ValueMatch:
    code: str
    label: str
    group: str
    score: float = 0.0
    alt_label: str = ""

@dataclass
class RetrievalResult:
    function_names: list[str]
    argument_values: dict[str, list[ValueMatch]]

# ===================== FunctionRetriever =====================
import unicodedata
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

class FunctionRetriever:
    def __init__(
        self,
        function_library: dict,
        method: Literal["bm25", "embedding", "hybrid"] = "hybrid",
        encoder_model: str = "sentence-transformers/all-MiniLM-L6-v2",
        bm25_weight: float = 0.4,
        emb_weight: float = 0.6,
        index_dir: str | None = None,
    ):
        self.library = function_library
        self.method = method
        self.func_names = list(function_library.keys())
        self.bm25_weight = bm25_weight
        self.emb_weight = emb_weight
        self.desc_list = [self._build_search_text(name, schema) for name, schema in function_library.items()]
        self._bm25 = None
        self._encoder = None
        self._embeddings = None
        if method in ("bm25", "hybrid"):
            self._init_bm25()
        if method in ("embedding", "hybrid"):
            self._init_embeddings(encoder_model, index_dir)

    @staticmethod
    def _build_search_text(name: str, schema: dict) -> str:
        parts = [name, schema.get("description", "")]
        for pname, pinfo in schema.get("parameters", {}).items():
            parts.append(pname)
            if isinstance(pinfo, dict):
                parts.append(pinfo.get("description", ""))
        parts.extend(schema.get("tags", []))
        return " ".join(p for p in parts if p)

    def _init_bm25(self):
        tokenized = [d.lower().split() for d in self.desc_list]
        self._bm25 = BM25Okapi(tokenized)

    def _init_embeddings(self, model_name: str, index_dir: str | None):
        self._encoder = SentenceTransformer(model_name)
        cache = Path(index_dir) / "func_embeddings.npy" if index_dir else None
        if cache and cache.exists():
            self._embeddings = np.load(str(cache))
        else:
            self._embeddings = self._encoder.encode(
                self.desc_list, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
            )
            if cache:
                cache.parent.mkdir(parents=True, exist_ok=True)
                np.save(str(cache), self._embeddings)

    def retrieve(self, query: str, k: int = 5) -> list[str]:
        scores = self._score(query)
        top_k = np.argsort(scores)[-k:][::-1]
        return [self.func_names[i] for i in top_k]

    def retrieve_with_scores(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        scores = self._score(query)
        top_k = np.argsort(scores)[-k:][::-1]
        return [(self.func_names[i], float(scores[i])) for i in top_k]

    def _score(self, query: str) -> np.ndarray:
        if self.method == "bm25":
            return self._bm25_scores(query)
        elif self.method == "embedding":
            return self._emb_scores(query)
        else:
            bm25 = self._minmax(self._bm25_scores(query))
            emb = self._minmax(self._emb_scores(query))
            return self.bm25_weight * bm25 + self.emb_weight * emb

    def _bm25_scores(self, query: str) -> np.ndarray:
        return np.array(self._bm25.get_scores(query.lower().split()), dtype=np.float32)

    def _emb_scores(self, query: str) -> np.ndarray:
        q = self._encoder.encode(query, convert_to_numpy=True, normalize_embeddings=True)
        return (self._embeddings @ q).astype(np.float32)

    @staticmethod
    def _minmax(arr: np.ndarray) -> np.ndarray:
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / (hi - lo + 1e-9)

# ===================== ArgumentValueRetriever =====================
class ArgumentValueRetriever:
    def __init__(self, argument_values: dict, top_k_values: int = 3):
        self.catalog = argument_values
        self.top_k_values = top_k_values
        self._norm_cache: dict[str, str] = {}

    def retrieve_for_function(self, query: str, function_schema: dict) -> dict[str, list[ValueMatch]]:
        result = {}
        params = function_schema.get("parameters", {})
        for param_name in params:
            values_for_param = self._get_catalog(param_name)
            if not values_for_param:
                continue
            matches = self._score_values(query, values_for_param)
            if matches:
                result[param_name] = matches
        return result

    def retrieve_for_functions(self, query: str, function_names: list[str], function_library: dict) -> dict[str, list[ValueMatch]]:
        combined = {}
        for fn in function_names:
            if fn not in function_library:
                continue
            fn_results = self.retrieve_for_function(query, function_library[fn])
            for param_name, matches in fn_results.items():
                if param_name not in combined:
                    combined[param_name] = matches
                else:
                    existing_codes = {m.code for m in combined[param_name]}
                    for m in matches:
                        if m.code not in existing_codes:
                            combined[param_name].append(m)
                    combined[param_name].sort(key=lambda x: -x.score)
                    combined[param_name] = combined[param_name][:self.top_k_values]
        return combined

    def _get_catalog(self, param_name: str) -> list[dict]:
        if param_name in self.catalog:
            return self.catalog[param_name]
        best_key = None
        best_len = 0
        for key in self.catalog:
            if param_name.endswith(key) and len(key) > best_len:
                best_key = key
                best_len = len(key)
        if best_key:
            return self.catalog[best_key]
        for key in self.catalog:
            if param_name.startswith(key):
                return self.catalog[key]
        return []

    def _score_values(self, query: str, catalog: list[dict]) -> list[ValueMatch]:
        query_norm = self._normalise(query)
        query_tokens = set(query_norm.split())
        scored = []
        for entry in catalog:
            code = str(entry.get("code", ""))
            label = str(entry.get("label", ""))
            alt_label = str(entry.get("alt_label", ""))
            group = str(entry.get("group", ""))
            score = self._match_score(query, query_norm, query_tokens, code, label, alt_label)
            if score > 0.0:
                scored.append(ValueMatch(code=code, label=label, group=group, score=score, alt_label=alt_label))
        scored.sort(key=lambda x: -x.score)
        return scored[:self.top_k_values]

    def _match_score(self, query: str, query_norm: str, query_tokens: set[str], code: str, label: str, alt_label: str) -> float:
        q_lower = query.lower()
        code_lower = code.lower()
        if code_lower in q_lower:
            specificity = min(1.0, len(code) / 5.0)
            return 0.8 + 0.2 * specificity
        label_norm = self._normalise(label)
        alt_label_norm = self._normalise(alt_label) if alt_label else ""
        if label_norm in query_norm:
            return 1.0
        if alt_label_norm and alt_label_norm in query_norm:
            return 0.9
        label_tokens = set(label_norm.split())
        alt_label_tokens = set(alt_label_norm.split()) if alt_label_norm else set()
        all_label_tokens = label_tokens | alt_label_tokens
        meaningful_label_tokens = {t for t in all_label_tokens if len(t) >= 2}
        meaningful_query_tokens = {t for t in query_tokens if len(t) >= 2}
        if not meaningful_label_tokens:
            return 0.0
        overlap = meaningful_query_tokens & meaningful_label_tokens
        if overlap:
            return min(0.7, 0.35 * len(overlap))
        return 0.0

    def _normalise(self, text: str) -> str:
        if text in self._norm_cache:
            return self._norm_cache[text]
        nfkd = unicodedata.normalize("NFKD", text.lower())
        ascii_text = "".join(c for c in nfkd if not unicodedata.combining(c))
        replacements = {"đ": "d", "ð": "d"}
        for src, dst in replacements.items():
            ascii_text = ascii_text.replace(src, dst)
        result = re.sub(r"\s+", " ", ascii_text).strip()
        self._norm_cache[text] = result
        return result

# ===================== Combined TelcoRetriever =====================
class TelcoRetriever:
    def __init__(self, function_retriever: FunctionRetriever, value_retriever: ArgumentValueRetriever):
        self.func_retriever = function_retriever
        self.value_retriever = value_retriever

    @classmethod
    def build(cls, function_library: dict, argument_values: dict, method: str = "hybrid",
              encoder_model: str = "sentence-transformers/all-MiniLM-L6-v2",
              top_k_values: int = 3, index_dir: str | None = None) -> "TelcoRetriever":
        func_ret = FunctionRetriever(function_library, method=method, encoder_model=encoder_model, index_dir=index_dir)
        val_ret = ArgumentValueRetriever(argument_values, top_k_values=top_k_values)
        return cls(func_ret, val_ret)

    def retrieve(self, query: str, function_library: dict, k: int = 5,
                 precomputed_func_names: list[str] | None = None) -> RetrievalResult:
        if precomputed_func_names is not None:
            func_names = precomputed_func_names
        else:
            func_names = self.func_retriever.retrieve(query, k=k)
        arg_values = self.value_retriever.retrieve_for_functions(query, func_names, function_library)
        return RetrievalResult(function_names=func_names, argument_values=arg_values)

In [ ]:
# ===================== excel_parser.py =====================
def _safe_json(value: str, fallback=None):
    if not isinstance(value, str) or not value.strip():
        return fallback
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(value)
        except Exception:
            return fallback

def _safe_list(value: str) -> list:
    result = _safe_json(value, fallback=None)
    if isinstance(result, list):
        return result
    if isinstance(value, str):
        return [v.strip() for v in value.replace("\n", ",").split(",") if v.strip()]
    return []

def parse_telecom_functions(excel_path: str, output_path: Optional[str] = "data/processed/function_library.json") -> dict:
    import ast
    path = Path(excel_path)
    if not path.exists():
        raise FileNotFoundError(f"Excel file not found: {excel_path}")
    df = pd.read_excel(path)
    required_cols = {"function_name", "description", "parameters"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Excel missing required columns: {missing}")
    library = {}
    for idx, row in df.iterrows():
        name = str(row["function_name"]).strip()
        if not name:
            continue
        params = _safe_json(row.get("parameters", "{}"), fallback={})
        examples = _safe_list(row.get("example_queries", "[]"))
        domain = _safe_json(row.get("domain_info", "{}"), fallback={})
        constraints = _safe_json(row.get("constraints", "{}"), fallback={})
        tags = _safe_list(row.get("tags", "[]"))
        func_schema = {
            "name": name,
            "description": str(row["description"]).strip(),
            "parameters": params,
            "examples": examples,
            "domain": domain,
            "constraints": constraints,
            "tags": tags,
        }
        library[name] = func_schema
    if output_path:
        out = Path(output_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        with open(out, "w", encoding="utf-8") as fh:
            json.dump(library, fh, indent=2, ensure_ascii=False)
        print(f"[excel_parser] Saved {len(library)} functions → {out}")
    return library

def load_function_library(library_path: str) -> dict:
    with open(library_path, "r", encoding="utf-8") as fh:
        return json.load(fh)

def load_function_schema(schema_path: str) -> dict:
    with open(schema_path, "r", encoding="utf-8") as fh:
        return json.load(fh)

In [ ]:
# ===================== base_reward.py =====================
_TOOL_CALL_RE = re.compile(r"<tool_call>(.*?)</tool_call>", re.DOTALL)
_REASONING_RE = re.compile(r"<reasoning>(.*?)</reasoning>", re.DOTALL)
_CALL_RE = _TOOL_CALL_RE
_REASON_RE = _REASONING_RE

def extract_call(response: str) -> dict | None:
    match = _TOOL_CALL_RE.search(response)
    if not match:
        old_re = re.compile(r"<call>(.*?)</call>", re.DOTALL)
        match = old_re.search(response)
    if not match:
        return None
    raw = match.group(1).strip()
    if raw.lower() == "null":
        return None
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return None

def extract_all_calls(response: str) -> list[dict]:
    calls = []
    for m in _TOOL_CALL_RE.finditer(response):
        try:
            parsed = json.loads(m.group(1).strip())
            if parsed is not None:
                calls.append(parsed)
        except json.JSONDecodeError:
            pass
    return calls

def extract_reasoning(response: str) -> str:
    match = _REASONING_RE.search(response)
    return match.group(1).strip() if match else ""

def schema_valid(response: str) -> float:
    return 1.0 if extract_call(response) is not None else 0.0

def func_selection_ok(response: str, expected_func: str) -> float:
    call = extract_call(response)
    if call is None:
        return 0.0
    return 1.0 if call.get("function") == expected_func else 0.0

def args_accuracy(response: str | dict, expected_args: dict) -> float:
    call = extract_call(response) if isinstance(response, str) else response
    if call is None:
        return 0.0
    if not expected_args:
        return 1.0
    actual = call.get("arguments", {})
    correct = sum(1 for k, v in expected_args.items() if str(actual.get(k, "")).strip() == str(v).strip())
    return correct / len(expected_args)

def reasoning_quality(response: str) -> float:
    text = extract_reasoning(response)
    if not text:
        return 0.0
    words = len(text.split())
    length_score = min(1.0, words / 50.0)
    has_steps = bool(re.search(r"(step\s*\d|first|then|finally|because|therefore|since)", text, re.IGNORECASE))
    return 0.7 * length_score + 0.3 * float(has_steps)

def format_reward(completions: list[str], **kwargs) -> list[float]:
    rewards = []
    for c in completions:
        has_tool_call = bool(_TOOL_CALL_RE.search(c))
        has_reasoning = bool(_REASONING_RE.search(c))
        score = 0.0
        if has_tool_call:
            score += 0.5
        if has_tool_call and has_reasoning:
            score += 0.5
        rewards.append(score)
    return rewards

In [ ]:
# ===================== gvpo_reward.py =====================
def process_reward_step(call: dict, ground_truth: dict, sandbox=None, args_threshold: float = 0.8) -> float:
    if not isinstance(call, dict) or not call:
        return 0.0
    if call.get("function") != ground_truth.get("function", ""):
        return 0.0
    if args_accuracy(call, ground_truth.get("arguments", {})) < args_threshold:
        return 0.0
    if sandbox is not None:
        if not sandbox.execute(call):
            return 0.0
    return 1.0

def outcome_reward(response: str, ground_truth: dict, sandbox=None, args_threshold: float = 0.8) -> float:
    calls = extract_all_calls(response)
    if not calls:
        return 0.0 if ground_truth.get("workflow") != "abstention" else 1.0
    for call in calls:
        if process_reward_step(call, ground_truth, sandbox, args_threshold) < 1.0:
            return 0.0
    return 1.0

def compute_process_shaping(response: str, ground_truth: dict, sandbox=None, args_threshold: float = 0.8) -> list[float]:
    calls = extract_all_calls(response)
    if not calls:
        return []
    step_rewards = [process_reward_step(call, ground_truth, sandbox, args_threshold) for call in calls]
    mu_proc = sum(step_rewards) / len(step_rewards)
    phi = [r - mu_proc for r in step_rewards]
    return phi

@dataclass
class GVPOTokenAdvantages:
    outcome_advantage: float
    per_token_shaped: torch.Tensor
    step_phis: list[float]
    step_process_rewards: list[float]

def _find_call_token_spans(response: str, tokenizer, max_seq_len: int) -> list[tuple[int, int]]:
    enc = tokenizer(response, return_offsets_mapping=True, truncation=True, max_length=max_seq_len, add_special_tokens=False)
    offsets = enc["offset_mapping"]
    spans = []
    for match in _TOOL_CALL_RE.finditer(response):
        char_start = match.start()
        char_end = match.end()
        tok_start = None
        tok_end = None
        for tok_idx, (off_s, off_e) in enumerate(offsets):
            if tok_start is None and off_e > char_start:
                tok_start = tok_idx
            if off_s < char_end:
                tok_end = tok_idx + 1
        if tok_start is not None and tok_end is not None:
            spans.append((tok_start, min(tok_end, max_seq_len)))
    return spans

def build_per_token_advantages(
    response: str, tokenizer, outcome_adv: float, ground_truth: dict,
    sandbox=None, shaping_coeff: float = 1.0, args_threshold: float = 0.8,
    max_seq_len: int = 512
) -> GVPOTokenAdvantages:
    phi_list = compute_process_shaping(response, ground_truth, sandbox, args_threshold)
    calls = extract_all_calls(response)
    raw_proc = [process_reward_step(c, ground_truth, sandbox, args_threshold) for c in calls]
    per_token = torch.full((max_seq_len,), fill_value=outcome_adv, dtype=torch.float32)
    if not phi_list:
        return GVPOTokenAdvantages(outcome_advantage=outcome_adv, per_token_shaped=per_token,
                                   step_phis=[], step_process_rewards=[])
    try:
        call_token_spans = _find_call_token_spans(response, tokenizer, max_seq_len)
    except Exception:
        return GVPOTokenAdvantages(outcome_advantage=outcome_adv, per_token_shaped=per_token,
                                   step_phis=phi_list, step_process_rewards=raw_proc)
    for step_idx, (tok_start, tok_end) in enumerate(call_token_spans):
        if step_idx >= len(phi_list):
            break
        shaped_value = outcome_adv + shaping_coeff * phi_list[step_idx]
        per_token[tok_start:tok_end] = shaped_value
    return GVPOTokenAdvantages(outcome_advantage=outcome_adv, per_token_shaped=per_token,
                               step_phis=phi_list, step_process_rewards=raw_proc)

def gvpo_reward_func(completions: list[str], ground_truth: list[dict] | None = None, **kwargs) -> list[float]:
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    return [outcome_reward(c, gt if isinstance(gt, dict) else {}) for c, gt in zip(completions, ground_truth)]

In [ ]:
# ===================== rc_grpo_reward.py =====================
HIGH_REWARD_TOKEN = "<|high_reward|>"
LOW_REWARD_TOKEN = "<|low_reward|>"

def rc_grpo_reward(response: str, ground_truth: dict, sandbox=None, args_threshold: float = 0.8) -> float:
    if not schema_valid(response):
        return 0.0
    if not func_selection_ok(response, ground_truth.get("function", "")):
        return 0.0
    if args_accuracy(response, ground_truth.get("arguments", {})) < args_threshold:
        return 0.0
    if sandbox is not None:
        if not sandbox.execute(response):
            return 0.0
    return 1.0

def rc_grpo_reward_func(completions: list[str], ground_truth: list[dict] | None = None, **kwargs) -> list[float]:
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    return [rc_grpo_reward(c, gt if isinstance(gt, dict) else {}) for c, gt in zip(completions, ground_truth)]

def rc_grpo_format_func(completions: list[str], **kwargs) -> list[float]:
    return format_reward(completions, **kwargs)

In [ ]:
# ===================== awpo_reward.py =====================
def _outcome_reward(response: str, ground_truth: dict, sandbox, outcome_weights: dict) -> float:
    fn_ok = func_selection_ok(response, ground_truth.get("function", ""))
    args_ok = args_accuracy(response, ground_truth.get("arguments", {}))
    if sandbox is not None:
        exec_ok = 1.0 if sandbox.execute(response) else 0.0
    else:
        exec_ok = schema_valid(response)
    return (outcome_weights["func_selection"] * fn_ok +
            outcome_weights["args_accuracy"] * args_ok +
            outcome_weights["execution"] * exec_ok)

def compute_group_stats(rewards: list[float]) -> dict:
    n = len(rewards)
    mean = sum(rewards) / n
    var = sum((r - mean) ** 2 for r in rewards) / n
    std = math.sqrt(var)
    return {"mean": mean, "var": var, "std": std}

def variance_gate(sigma2_out: float, tau: float = 0.5) -> float:
    return 0.0 if sigma2_out > tau else 1.0

def difficulty_weight(mu_out: float) -> float:
    return 4.0 * mu_out * (1.0 - mu_out)

def awpo_group_advantages(outcome_rewards: list[float], reasoning_rewards: list[float],
                          tau: float = 0.5, eps: float = 1e-8) -> tuple[list[float], float]:
    n = len(outcome_rewards)
    out_stats = compute_group_stats(outcome_rewards)
    mu_out = out_stats["mean"]
    sigma2_out = out_stats["var"]
    g = variance_gate(sigma2_out, tau)
    w = difficulty_weight(mu_out)
    mixed = [(1.0 - g) * r_out + g * r_rea for r_out, r_rea in zip(outcome_rewards, reasoning_rewards)]
    mix_stats = compute_group_stats(mixed)
    mu_mix = mix_stats["mean"]
    std_mix = mix_stats["std"]
    advantages = [w * (r - mu_mix) / (std_mix + eps) for r in mixed]
    adaptive_clip_delta = g * 0.2
    return advantages, adaptive_clip_delta

def awpo_reward(response: str, ground_truth: dict, sandbox=None,
                group_stats: dict | None = None, outcome_weights: dict | None = None,
                tau: float = 0.5) -> float:
    ow = outcome_weights or {"func_selection": 0.4, "args_accuracy": 0.3, "execution": 0.3}
    r_out = _outcome_reward(response, ground_truth, sandbox, ow)
    r_reason = reasoning_quality(response)
    if group_stats is None:
        return r_out
    sigma2_out = group_stats.get("var", 1.0)
    g = variance_gate(sigma2_out, tau)
    return (1.0 - g) * r_out + g * r_reason

def awpo_reward_func(completions: list[str], ground_truth: list[dict] | None = None, **kwargs) -> list[float]:
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    outcome_weights = {"func_selection": 0.4, "args_accuracy": 0.3, "execution": 0.3}
    out_rewards = [_outcome_reward(c, gt if isinstance(gt, dict) else {}, None, outcome_weights)
                   for c, gt in zip(completions, ground_truth)]
    return out_rewards

In [ ]:
# ===================== base_trainer.py =====================
CUSTOM_CHAT_TEMPLATE = (
    "{%- for message in messages %}"
    "{%- if message.role == 'system' %}"
    "{{- '<|im_start|>system\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'user' %}"
    "{{- '<|im_start|>user\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'assistant' %}"
    "{{- '<|im_start|>assistant\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'tool' %}"
    "{{- '<|im_start|>tool\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- elif message.role == 'retriever' %}"
    "{{- '<|im_start|>retriever\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- else %}"
    "{{- '<|im_start|>' + message.role + '\n' + message.content + '\n<|im_end|>\n' }}"
    "{%- endif %}"
    "{%- endfor %}"
    "{%- if add_generation_prompt %}"
    "{{- '<|im_start|>assistant\n' }}"
    "{%- endif %}"
)

def patch_tokenizer_for_custom_roles(tokenizer) -> None:
    tokenizer.chat_template = CUSTOM_CHAT_TEMPLATE
    print("[base_trainer] Custom chat template registered (supports 'retriever' role).")

SYSTEM_PROMPT = """\
You are a telecom network operations assistant. \
You will receive a user query, a list of available functions with their \
parameter schemas, and relevant argument values retrieved from a knowledge base.

Your task:
1. Read the user query carefully
2. Review the available functions and retrieved argument values in the \
<|im_start|>retriever<|im_end|> block
3. Reason step by step about which function to call and what argument \
values to use
4. Output your reasoning and the final function call

REQUIRED OUTPUT FORMAT — use these exact tags:
<reasoning>
Step-by-step analysis:
- What the user is asking for
- Which function best matches and why
- Which argument values from the retriever match the query
- Final argument assignments
</reasoning>
<tool_call>
{"function": "<function_name>", "arguments": {"<param1>": "<value1>", "<param2>": "<value2>"}}
</tool_call>

STRICT RULES:
- Call EXACTLY one function from the retriever list (no more, no less)
- Include ALL required parameters
- Use argument values from the retriever when they match the query
- Do NOT invent functions or parameters not in the retriever block
- If the query is unanswerable with available functions, output:
<reasoning>Explanation of why no function is appropriate.</reasoning>
<tool_call>null</tool_call>
"""

def build_function_description(func_name: str, schema: dict) -> str:
    lines = [f"### {func_name}"]
    lines.append(f"Description: {schema.get('description', 'No description available')}")
    params = schema.get("parameters", {})
    if params:
        lines.append("Parameters:")
        for pname, pinfo in params.items():
            if isinstance(pinfo, dict):
                ptype = pinfo.get("type", "any")
                required = "(required)" if pinfo.get("required") else "(optional)"
                desc = pinfo.get("description", "")
                lines.append(f"  - {pname} [{ptype}] {required}: {desc}")
            else:
                lines.append(f"  - {pname}: {pinfo}")
    constraints = schema.get("constraints", {})
    if constraints:
        lines.append(f"Constraints: {json.dumps(constraints, ensure_ascii=False)}")
    return "\n".join(lines)

def build_argument_values_block(argument_values: dict[str, list]) -> str:
    if not argument_values:
        return ""
    lines = ["## Relevant Argument Values"]
    for param_name, matches in argument_values.items():
        if not matches:
            continue
        lines.append(f"\n### {param_name}")
        for m in matches:
            if hasattr(m, "code"):
                label_str = m.label
                if m.alt_label:
                    label_str += f" / {m.alt_label}"
                lines.append(f"  - {m.code} → {label_str}  [{m.group}]")
            else:
                code = m.get("code", "")
                label = m.get("label", "")
                group = m.get("group", "")
                lines.append(f"  - {code} → {label}  [{group}]")
    return "\n".join(lines)

def build_retriever_block(function_names: list[str], function_library: dict,
                          argument_values: dict[str, list] | None = None) -> str:
    lines = ["## Available Functions\n"]
    for fn in function_names:
        if fn in function_library:
            lines.append(build_function_description(fn, function_library[fn]))
            lines.append("")
    if argument_values:
        val_block = build_argument_values_block(argument_values)
        if val_block:
            lines.append(val_block)
    return "\n".join(lines)

def build_messages_for_grpo(query: str, function_names: list[str], function_library: dict,
                            argument_values: dict[str, list] | None = None) -> list[dict]:
    retriever_content = build_retriever_block(function_names, function_library, argument_values)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query},
        {"role": "retriever", "content": retriever_content},
    ]

def build_messages_for_sft(query: str, function_names: list[str], function_library: dict,
                           ground_truth: dict, argument_values: dict[str, list] | None = None) -> list[dict]:
    messages = build_messages_for_grpo(query, function_names, function_library, argument_values)
    reasoning = ground_truth.get("reasoning", "Analysing the query to determine the correct function and arguments.")
    func_name = ground_truth.get("function")
    arguments = ground_truth.get("arguments", {})
    call_json = "null" if func_name is None else json.dumps({"function": func_name, "arguments": arguments}, indent=2, ensure_ascii=False)
    assistant_content = f"<reasoning>\n{reasoning}\n</reasoning>\n<tool_call>\n{call_json}\n</tool_call>"
    messages.append({"role": "assistant", "content": assistant_content})
    return messages

def format_sample_for_grpo(sample: dict, function_library: dict, tokenizer,
                           argument_values: dict[str, list] | None = None) -> dict:
    query = sample["query"]
    retrieved = sample.get("retrieved_functions", [])
    gt = sample.get("ground_truth", {})
    arg_vals = argument_values or sample.get("retrieved_argument_values")
    messages = build_messages_for_grpo(query, retrieved, function_library, arg_vals)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt, "ground_truth": gt, "query": query, "workflow_type": sample.get("workflow_type", "single_call")}

def format_sample_for_sft(sample: dict, function_library: dict, tokenizer,
                          argument_values: dict[str, list] | None = None) -> dict:
    query = sample["query"]
    retrieved = sample.get("retrieved_functions", [])
    gt = sample.get("ground_truth", {})
    arg_vals = argument_values or sample.get("retrieved_argument_values")
    messages = build_messages_for_sft(query, retrieved, function_library, gt, arg_vals)
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

def load_grpo_dataset(jsonl_path: str, function_library: dict, tokenizer,
                      argument_values_catalog: dict | None = None, telco_retriever=None) -> "Dataset":
    if Dataset is None or jsonlines is None:
        raise ImportError("datasets and/or jsonlines not installed.")
    raw_samples = []
    with jsonlines.open(jsonl_path) as reader:
        for obj in reader:
            raw_samples.append(obj)
    print(f"[base_trainer] Loaded {len(raw_samples)} samples from {jsonl_path}")
    val_retriever = None
    if telco_retriever is None and argument_values_catalog is not None:
        if ArgumentValueRetriever is not None:
            val_retriever = ArgumentValueRetriever(argument_values_catalog)
    formatted = []
    for sample in raw_samples:
        query = sample["query"]
        retrieved = sample.get("retrieved_functions", [])
        if telco_retriever is not None:
            result = telco_retriever.retrieve(query, function_library, precomputed_func_names=retrieved)
            arg_vals = result.argument_values
        elif val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(query, retrieved, function_library)
        else:
            arg_vals = sample.get("retrieved_argument_values")
        formatted.append(format_sample_for_grpo(sample, function_library, tokenizer, arg_vals))
    dataset = Dataset.from_list(formatted)
    print(f"[base_trainer] GRPO dataset ready: {len(dataset)} samples")
    return dataset

def load_sft_dataset(jsonl_path: str, function_library: dict, tokenizer,
                     argument_values_catalog: dict | None = None, telco_retriever=None) -> "Dataset":
    if Dataset is None or jsonlines is None:
        raise ImportError("datasets and/or jsonlines not installed.")
    raw_samples = []
    with jsonlines.open(jsonl_path) as reader:
        for obj in reader:
            raw_samples.append(obj)
    val_retriever = None
    if telco_retriever is None and argument_values_catalog is not None:
        if ArgumentValueRetriever is not None:
            val_retriever = ArgumentValueRetriever(argument_values_catalog)
    formatted = []
    for sample in raw_samples:
        query = sample["query"]
        retrieved = sample.get("retrieved_functions", [])
        if telco_retriever is not None:
            result = telco_retriever.retrieve(query, function_library, precomputed_func_names=retrieved)
            arg_vals = result.argument_values
        elif val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(query, retrieved, function_library)
        else:
            arg_vals = sample.get("retrieved_argument_values")
        formatted.append(format_sample_for_sft(sample, function_library, tokenizer, arg_vals))
    dataset = Dataset.from_list(formatted)
    print(f"[base_trainer] SFT dataset ready: {len(dataset)} samples")
    return dataset

In [ ]:
# ===================== model loading function =====================
def load_model(config: dict | None = None):
    from unsloth import FastLanguageModel, PatchFastRL
    PatchFastRL("GRPO", FastLanguageModel)
    cfg = config or {}
    model_cfg = cfg.get("model", {})
    lora_cfg = cfg.get("lora", {})
    model_name = model_cfg.get("name", "unsloth/Qwen3-4B-Base")
    max_seq = model_cfg.get("max_seq_length", 2048)
    lora_rank = lora_cfg.get("r", 16)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=max_seq,
        load_in_4bit=model_cfg.get("load_in_4bit", True),
        dtype=None,
        fast_inference=model_cfg.get("fast_inference", True),
        max_lora_rank=lora_rank,
        gpu_memory_utilization=model_cfg.get("gpu_memory_utilization", 0.7),
    )
    patch_tokenizer_for_custom_roles(tokenizer)
    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_rank,
        lora_alpha=lora_cfg.get("lora_alpha", 2 * lora_rank),
        lora_dropout=lora_cfg.get("lora_dropout", 0.0),
        target_modules=lora_cfg.get("target_modules", [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ]),
        bias=lora_cfg.get("bias", "none"),
        use_gradient_checkpointing=lora_cfg.get("use_gradient_checkpointing", "unsloth"),
        random_state=3407,
    )
    return model, tokenizer

def build_grpo_config(config: dict, output_dir: str | None = None) -> GRPOConfig:
    from trl import GRPOConfig
    from vllm import SamplingParams
    train_cfg = config.get("training", {})
    grpo_cfg = config.get("grpo", {})
    data_cfg = config.get("data", {})
    vllm_params = SamplingParams(
        temperature=grpo_cfg.get("temperature", 1.0),
        top_p=0.95,
        min_p=0.05,
        seed=train_cfg.get("seed", 3407),
        stop=["</tool_call>"],
        include_stop_str_in_output=True,
    )
    return GRPOConfig(
        output_dir=output_dir or train_cfg.get("output_dir", "outputs/model"),
        learning_rate=train_cfg.get("learning_rate", 5e-6),
        adam_beta1=train_cfg.get("adam_beta1", 0.9),
        adam_beta2=train_cfg.get("adam_beta2", 0.99),
        weight_decay=train_cfg.get("weight_decay", 0.01),
        warmup_ratio=train_cfg.get("warmup_ratio", 0.1),
        lr_scheduler_type=train_cfg.get("lr_scheduler_type", "cosine"),
        optim=train_cfg.get("optim", "adamw_8bit"),
        per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 1),
        gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 4),
        num_generations=train_cfg.get("num_generations", 8),
        max_prompt_length=data_cfg.get("max_prompt_length", 1024),
        max_completion_length=data_cfg.get("max_completion_length", 512),
        temperature=grpo_cfg.get("temperature", 1.0),
        epsilon=grpo_cfg.get("epsilon", 0.2),
        epsilon_high=grpo_cfg.get("epsilon_high", 0.28),
        loss_type=grpo_cfg.get("loss_type", "grpo"),
        mask_truncated_completions=grpo_cfg.get("mask_truncated_completions", True),
        vllm_sampling_params=vllm_params,
        max_steps=train_cfg.get("max_steps", 500),
        save_steps=train_cfg.get("save_steps", 100),
        logging_steps=train_cfg.get("logging_steps", 1),
        max_grad_norm=train_cfg.get("max_grad_norm", 0.1),
        report_to=train_cfg.get("report_to", "none"),
        seed=train_cfg.get("seed", 3407),
        bf16=torch.cuda.is_bf16_supported(),
    )

In [ ]:
# ===================== gvpo_trainer.py =====================
class GVPOTrainer(GRPOTrainer):
    def __init__(self, *args, gvpo_config: dict | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        cfg = gvpo_config or {}
        self.shaping_coeff = cfg.get("shaping_coeff", 1.0)
        self.args_threshold = cfg.get("args_threshold", 0.8)
        self._current_ground_truths = []
        self._current_completions = []
        logger = get_logger(__name__)
        logger.info(f"[GVPO] shaping_coeff b={self.shaping_coeff}  args_threshold={self.args_threshold}")

    def _cache_inputs(self, completions: list[str], ground_truths: list[dict]) -> None:
        self._current_completions = completions
        self._current_ground_truths = ground_truths

    def compute_loss(self, model, inputs: dict[str, Any], return_outputs: bool = False,
                     num_items_in_batch: int | None = None):
        shaped = self._shape_advantages(inputs)
        if shaped is not None:
            inputs["advantages"] = shaped
        return super().compute_loss(model, inputs, return_outputs, num_items_in_batch)

    def _shape_advantages(self, inputs: dict[str, Any]) -> torch.Tensor | None:
        flat_advs = inputs.get("advantages")
        attention_mask = inputs.get("attention_mask")
        completions = self._current_completions
        ground_truths = self._current_ground_truths
        if flat_advs is None or attention_mask is None or not completions or not ground_truths:
            return None
        B, T = attention_mask.shape
        if len(completions) != B:
            return None
        device = flat_advs.device
        if flat_advs.dim() == 1:
            flat_advs_2d = flat_advs.unsqueeze(1).expand(B, T).clone()
        else:
            flat_advs_2d = flat_advs.clone()
        shaped_advs = flat_advs_2d.clone()
        for i, (completion, gt) in enumerate(zip(completions, ground_truths)):
            if not isinstance(gt, dict):
                continue
            outcome_adv = float(flat_advs[i] if flat_advs.dim() == 1 else flat_advs[i].mean())
            phi_list = compute_process_shaping(completion, gt, sandbox=None, args_threshold=self.args_threshold)
            if not phi_list:
                continue
            try:
                call_spans = _find_call_token_spans(completion, self.processing_class, T)
            except Exception:
                continue
            for step_idx, (tok_start, tok_end) in enumerate(call_spans):
                if step_idx >= len(phi_list):
                    break
                shaped_value = outcome_adv + self.shaping_coeff * phi_list[step_idx]
                shaped_advs[i, tok_start:tok_end] = shaped_value
        return shaped_advs.to(device)

    def _compute_rewards(self, *args, **kwargs):
        result = super()._compute_rewards(*args, **kwargs)
        try:
            completions = kwargs.get("completions", args[0] if args else [])
            ground_truths = kwargs.get("ground_truth", [])
            if completions and ground_truths:
                self._cache_inputs(completions, ground_truths)
        except Exception:
            pass
        return result

# ===================== rc_grpo_trainer.py =====================
def inject_diverse_reward_tokens(dataset, num_generations: int = 8,
                                 high_token: str = HIGH_REWARD_TOKEN,
                                 low_token: str = LOW_REWARD_TOKEN,
                                 high_fraction: float = 0.5):
    import random
    n_high = max(1, round(num_generations * high_fraction))
    def _expand(example, idx):
        out = []
        for g in range(num_generations):
            ex = dict(example)
            token = high_token if g < n_high else low_token
            if not ex["prompt"].startswith(token):
                ex["prompt"] = token + "\n" + ex["prompt"]
            ex["reward_token"] = token
            out.append(ex)
        rnd = random.Random(int(idx))
        rnd.shuffle(out)
        return out
    return dataset.flat_map(_expand, with_indices=True)

def compute_high_fraction_from_rctp_dataset(rctp_dataset_path: str) -> float:
    total = 0
    high = 0
    with jsonlines.open(rctp_dataset_path) as reader:
        for obj in reader:
            total += 1
            reward = obj.get("reward", 0)
            if reward == 1:
                high += 1
    return high / max(total, 1)

class RCGRPOTrainer(GRPOTrainer):
    @classmethod
    def register_reward_tokens(cls, tokenizer) -> None:
        new_tokens = [HIGH_REWARD_TOKEN, LOW_REWARD_TOKEN]
        existing = set(tokenizer.additional_special_tokens)
        to_add = [t for t in new_tokens if t not in existing]
        if to_add:
            tokenizer.add_special_tokens({"additional_special_tokens": to_add})
            logger = get_logger(__name__)
            logger.info(f"[RC-GRPO] Registered special tokens: {to_add}")

# ===================== autotool_trainer.py =====================
def autotool_reward_func(completions: list[str], ground_truth: list[dict] | None = None, **kwargs) -> list[float]:
    if ground_truth is None:
        ground_truth = kwargs.get("ground_truths", [{}] * len(completions))
    rewards = []
    for comp, gt in zip(completions, ground_truth):
        if not isinstance(gt, dict):
            gt = {}
        ok = (schema_valid(comp) and func_selection_ok(comp, gt.get("function", "")) == 1.0
              and args_accuracy(comp, gt.get("arguments", {})) >= 0.8)
        outcome = 1.0 if ok else 0.0
        workflow_type = gt.get("workflow", "single_call")
        length = len(comp.split())
        if workflow_type == "single_call":
            penalty = max(0.0, (length - 100) / 200.0)
        elif workflow_type == "parallel":
            penalty = max(0.0, (length - 150) / 300.0)
        else:
            penalty = 0.0
        reward = max(0.0, min(1.0, outcome - 0.1 * penalty))
        rewards.append(reward)
    return rewards

class AutoToolTrainer(GRPOTrainer):
    def __init__(self, *args, autotool_config: dict | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        cfg = autotool_config or {}
        self.entropy_coeff_reasoning = cfg.get("entropy_coeff_reasoning", 0.01)
        self.entropy_coeff_tool = cfg.get("entropy_coeff_tool", 0.005)
        self.entropy_coeff_other = cfg.get("entropy_coeff_other", 0.001)
        logger = get_logger(__name__)
        logger.info(f"[AutoTool] entropy_coeff_reasoning={self.entropy_coeff_reasoning}  tool={self.entropy_coeff_tool}  other={self.entropy_coeff_other}")

    def compute_loss(self, model, inputs: dict[str, Any], return_outputs: bool = False,
                     num_items_in_batch: int | None = None):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, num_items_in_batch=num_items_in_batch)
        logits = outputs.logits if hasattr(outputs, "logits") else None
        if logits is None:
            return loss if not return_outputs else (loss, outputs)
        attention_mask = inputs.get("attention_mask")
        input_ids = inputs.get("input_ids")
        if input_ids is None or attention_mask is None:
            return loss if not return_outputs else (loss, outputs)
        tokenizer = self.processing_class
        try:
            reasoning_start = tokenizer.convert_tokens_to_ids("<reasoning>")
            reasoning_end = tokenizer.convert_tokens_to_ids("</reasoning>")
            call_start = tokenizer.convert_tokens_to_ids("<tool_call>")
            call_end = tokenizer.convert_tokens_to_ids("</tool_call>")
        except Exception:
            reasoning_start = reasoning_end = call_start = call_end = None
        if reasoning_start is None:
            coeff = self.entropy_coeff_other
            probs = F.softmax(logits, dim=-1)
            entropy = -torch.sum(probs * F.log_softmax(logits, dim=-1), dim=-1)
            entropy_loss = -coeff * (entropy * attention_mask).sum() / attention_mask.sum()
            loss = loss + entropy_loss
            return loss if not return_outputs else (loss, outputs)
        B, T = input_ids.shape
        reasoning_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        tool_mask = torch.zeros_like(input_ids, dtype=torch.bool)
        for b in range(B):
            in_reasoning = False
            in_tool = False
            for t in range(T):
                token_id = input_ids[b, t].item()
                if token_id == reasoning_start:
                    in_reasoning = True
                elif token_id == reasoning_end:
                    in_reasoning = False
                elif token_id == call_start:
                    in_tool = True
                elif token_id == call_end:
                    in_tool = False
                if in_reasoning:
                    reasoning_mask[b, t] = True
                elif in_tool:
                    tool_mask[b, t] = True
        probs = F.softmax(logits, dim=-1)
        log_probs = F.log_softmax(logits, dim=-1)
        entropy = -torch.sum(probs * log_probs, dim=-1)
        mask = attention_mask.bool()
        entropy_reasoning = (entropy * reasoning_mask * mask).sum()
        entropy_tool = (entropy * tool_mask * mask).sum()
        entropy_other = (entropy * (~(reasoning_mask | tool_mask)) * mask).sum()
        total_tokens = mask.sum().float()
        entropy_loss = -(
            self.entropy_coeff_reasoning * entropy_reasoning / (mask.sum() + 1e-8)
            + self.entropy_coeff_tool * entropy_tool / (mask.sum() + 1e-8)
            + self.entropy_coeff_other * entropy_other / (mask.sum() + 1e-8)
        )
        loss = loss + entropy_loss
        if not return_outputs:
            return loss
        return loss, outputs

# ===================== awpo_trainer.py =====================
class AWPOTrainer(GRPOTrainer):
    def __init__(self, *args, awpo_config: dict | None = None, **kwargs):
        super().__init__(*args, **kwargs)
        cfg = awpo_config or {}
        self.clip_shrink_coeff = cfg.get("clip_shrink_coeff", 0.2)
        self.outcome_weights = cfg.get("outcome_weights", {"func_selection": 0.4, "args_accuracy": 0.3, "execution": 0.3})
        self._reasoning_cache = {}
        self._last_mixing_weight = 0.0
        logger = get_logger(__name__)
        logger.info(f"[AWPO] clip_shrink_coeff η={self.clip_shrink_coeff}")

    def _generate_and_score_completions(self, prompts, **kwargs):
        completions, rewards = super()._generate_and_score_completions(prompts, **kwargs)
        for comp in completions:
            if comp not in self._reasoning_cache:
                self._reasoning_cache[comp] = reasoning_quality(comp)
        return completions, rewards

    def compute_advantages(self, rewards: torch.Tensor, group_indices: torch.Tensor,
                           completions: list[str] | None = None) -> tuple[torch.Tensor, float]:
        B = rewards.shape[0]
        advantages = torch.zeros(B, dtype=torch.float32)
        mixing_weights = []
        unique_groups = group_indices.unique().tolist()
        eps = 1e-8
        for gid in unique_groups:
            mask = (group_indices == gid).nonzero(as_tuple=True)[0]
            g_idx = mask.tolist()
            out_r = [float(rewards[i]) for i in g_idx]
            if completions is not None:
                reason_r = [self._reasoning_cache.get(completions[i], 0.0) for i in g_idx]
            else:
                reason_r = [0.0] * len(g_idx)
            n = len(out_r)
            mu_out = sum(out_r) / n
            sigma2_out = sum((r - mu_out) ** 2 for r in out_r) / n
            sigma_out = math.sqrt(sigma2_out + eps)
            rho = max(0.0, min(1.0, 0.5 * (1.0 - sigma_out)))
            for _ in range(10):
                mixed = [(1 - rho) * r_out + rho * r_rea for r_out, r_rea in zip(out_r, reason_r)]
                mu_mix = sum(mixed) / n
                sigma_mix = math.sqrt(sum((r - mu_mix) ** 2 for r in mixed) / n + eps)
                new_rho = sigma_mix / (sigma_out + sigma_mix + eps)
                if abs(new_rho - rho) < 1e-4:
                    rho = new_rho
                    break
                rho = new_rho
            w = 4.0 * mu_out * (1.0 - mu_out)
            mixed = [(1 - rho) * r_out + rho * r_rea for r_out, r_rea in zip(out_r, reason_r)]
            mu_mix = sum(mixed) / n
            sigma_mix = math.sqrt(sum((r - mu_mix) ** 2 for r in mixed) / n + eps)
            sigma_mix = max(sigma_mix, eps)
            group_advs = [w * (r - mu_mix) / sigma_mix for r in mixed]
            for local_idx, global_idx in enumerate(g_idx):
                advantages[global_idx] = group_advs[local_idx]
            mixing_weights.append(rho)
        self._last_mixing_weight = sum(mixing_weights) / max(len(mixing_weights), 1)
        return advantages, self._last_mixing_weight

    def compute_loss(self, model, inputs: dict[str, Any], return_outputs: bool = False,
                     num_items_in_batch: int | None = None):
        rho = getattr(self, "_last_mixing_weight", 0.0)
        original_eps = self.args.epsilon
        if rho > 0.0:
            self.args.epsilon = original_eps / (1.0 + self.clip_shrink_coeff * rho)
        loss_output = super().compute_loss(model, inputs, return_outputs, num_items_in_batch)
        self.args.epsilon = original_eps
        return loss_output

In [ ]:
# ===================== metrics.py =====================
def compute_all_metrics(response: str, ground_truth: dict, sandbox, latency_ms: float,
                        cost_estimate: float, function_library: dict) -> dict[str, float]:
    call = extract_call(response)
    expected_func = ground_truth.get("function", "none")
    expected_args = ground_truth.get("arguments", {})
    workflow = ground_truth.get("workflow", "single_call")
    func_sel_acc = func_selection_ok(response, expected_func)
    arg_acc = args_accuracy(response, expected_args)
    schema_val = schema_valid(response)
    exec_success = 0.0
    if sandbox is not None:
        exec_success = 1.0 if sandbox.execute(response) else 0.0
    else:
        exec_success = float(call is not None)
    task_success = float(func_sel_acc == 1.0 and arg_acc >= 0.8 and exec_success == 1.0)
    hallucinated = 0.0
    if call is not None:
        called_fn = call.get("function", "")
        if called_fn and called_fn not in function_library:
            hallucinated = 1.0
    abstention_acc = float("nan")
    if workflow == "abstention":
        produced_call = extract_call(response)
        abstention_acc = 1.0 if (produced_call is None or produced_call == "null") else 0.0
    latency = latency_ms
    cost = cost_estimate
    return {
        "function_selection_accuracy": func_sel_acc,
        "argument_accuracy": arg_acc,
        "schema_validity": schema_val,
        "execution_success_rate": exec_success,
        "task_success_rate": task_success,
        "hallucinated_call_rate": hallucinated,
        "abstention_accuracy": abstention_acc,
        "latency_ms": latency,
        "cost_per_query_usd": cost,
    }

def aggregate_metrics(results: list[dict[str, float]]) -> dict[str, float]:
    if not results:
        return {}
    keys = results[0].keys()
    agg = {}
    for k in keys:
        vals = [r[k] for r in results if not np.isnan(r[k])]
        if vals:
            agg[k] = float(np.mean(vals))
            agg[f"{k}__std"] = float(np.std(vals))
            agg[f"{k}__count"] = len(vals)
        else:
            agg[k] = float("nan")
    return agg

def estimate_cost(prompt: str, response: str, price_per_1k_tokens: float = 0.0002) -> float:
    total_chars = len(prompt) + len(response)
    tokens_est = total_chars / 1.3
    return (tokens_est / 1000) * price_per_1k_tokens

# ===================== benchmark.py =====================
def evaluate_model(
    model_path: str,
    test_dataset_path: str,
    function_library: dict,
    retriever: FunctionRetriever,
    sandbox: Sandbox,
    top_k: int = 5,
    max_new_tokens: int = 512,
    model_name_tag: str = "model",
    use_dataset_retrieval: bool = True,
    argument_values: dict | None = None,
) -> dict:
    logger = get_logger(__name__)
    logger.info(f"[Benchmark] Loading model from {model_path}")
    model, tokenizer = load_model_from_path(model_path)
    patch_tokenizer_for_custom_roles(tokenizer)
    val_retriever = None
    if argument_values is not None:
        val_retriever = ArgumentValueRetriever(argument_values)
    test_samples = []
    with jsonlines.open(test_dataset_path) as reader:
        for obj in reader:
            test_samples.append(obj)
    logger.info(f"[Benchmark] {model_name_tag}: evaluating {len(test_samples)} samples")
    results = []
    for sample in tqdm(test_samples, desc=f"Eval [{model_name_tag}]"):
        query = sample["query"]
        gt = sample.get("ground_truth", {})
        if use_dataset_retrieval and sample.get("retrieved_functions"):
            retrieved = sample["retrieved_functions"]
        else:
            retrieved = retriever.retrieve(query, k=top_k)
        if val_retriever is not None:
            arg_vals = val_retriever.retrieve_for_functions(query, retrieved, function_library)
        else:
            arg_vals = sample.get("retrieved_argument_values")
        messages = build_messages_for_grpo(query, retrieved, function_library, arg_vals)
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        t0 = time.perf_counter()
        response = generate_response(model, tokenizer, prompt, max_new_tokens)
        latency = (time.perf_counter() - t0) * 1000.0
        cost = estimate_cost(prompt, response)
        metrics = compute_all_metrics(response, gt, sandbox, latency, cost, function_library)
        metrics["sample_id"] = sample.get("id", "")
        results.append(metrics)
    agg = aggregate_metrics(results)
    logger.info(f"[Benchmark] {model_name_tag} aggregate: {agg}")
    return {"model": model_name_tag, "per_sample": results, "aggregate": agg}

In [ ]:
# ===================== report_generator.py =====================
METRIC_DISPLAY_NAMES = {
    "function_selection_accuracy": "Func. Selection Acc.",
    "argument_accuracy": "Arg. Accuracy",
    "schema_validity": "Schema Validity",
    "execution_success_rate": "Exec. Success Rate",
    "task_success_rate": "Task Success Rate",
    "hallucinated_call_rate": "Hallucination Rate ↓",
    "abstention_accuracy": "Abstention Acc.",
    "latency_ms": "Latency (ms) ↓",
    "cost_per_query_usd": "Cost/Query (USD) ↓",
}

def generate_report(eval_results: list[dict], output_dir: str = "outputs/evaluation_reports") -> None:
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for result in eval_results:
        model = result["model"]
        agg = result["aggregate"]
        row = {"Model": model}
        for k, display in METRIC_DISPLAY_NAMES.items():
            row[display] = round(agg.get(k, float("nan")), 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Model")
    print("\n" + "=" * 80)
    print("  EVALUATION REPORT — Telecom Tool-Calling with RL")
    print("=" * 80)
    print(tabulate(df, headers="keys", tablefmt="github", floatfmt=".4f"))
    csv_path = out / "metrics_summary.csv"
    df.to_csv(csv_path)
    print(f"[Report] CSV saved → {csv_path}")
    json_path = out / "full_results.json"
    with open(json_path, "w") as fh:
        json.dump(eval_results, fh, indent=2, default=str)
    print(f"[Report] JSON saved → {json_path}")
    # Optional plots
    try:
        _plot_bar_comparison(df, out)
        _plot_radar(df, out)
    except Exception as e:
        print(f"Plot generation failed: {e}")

def _plot_bar_comparison(df: pd.DataFrame, out: Path) -> None:
    core_metrics = ["Func. Selection Acc.", "Arg. Accuracy", "Schema Validity",
                    "Exec. Success Rate", "Task Success Rate"]
    plot_df = df[[c for c in core_metrics if c in df.columns]]
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_df.T.plot(kind="bar", ax=ax, width=0.7, colormap="tab10")
    ax.set_title("Model Comparison – Core Metrics", fontsize=14, fontweight="bold")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Model", loc="lower right")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    fig.savefig(out / "bar_comparison.png", dpi=150)
    plt.close(fig)

def _plot_radar(df: pd.DataFrame, out: Path) -> None:
    radar_metrics = ["Func. Selection Acc.", "Arg. Accuracy", "Schema Validity",
                     "Exec. Success Rate", "Task Success Rate", "Abstention Acc."]
    categories = [c for c in radar_metrics if c in df.columns]
    N = len(categories)
    if N < 3:
        return
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    fig, ax = plt.subplots(1, 1, figsize=(8, 8), subplot_kw={"polar": True})
    cmap = plt.get_cmap("tab10")
    for i, (model, row) in enumerate(df.iterrows()):
        vals = [row.get(c, 0.0) for c in categories]
        vals += vals[:1]
        ax.plot(angles, vals, "o-", linewidth=2, color=cmap(i), label=model)
        ax.fill(angles, vals, alpha=0.1, color=cmap(i))
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_ylim(0, 1)
    ax.set_title("Algorithm Radar Comparison", size=14, fontweight="bold", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    fig.savefig(out / "radar_comparison.png", dpi=150)
    plt.close(fig)

In [ ]:
# ===================== CONFIGURATION =====================
# Choose which algorithm to run: 'grpo', 'gvpo', 'rc_grpo', 'autotool', 'awpo'
ALGORITHM = "rc_grpo"  # change as needed

# Data paths (upload your files to Colab or mount drive)
DATA_DIR = Path("data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# You need to have function_library.json and argument_values.json
# Either upload them, or generate from Excel using parse_telecom_functions.
# For demo, we'll assume you have them.
FUNCTION_LIBRARY_PATH = DATA_DIR / "function_library.json"
ARGUMENT_VALUES_PATH = DATA_DIR / "argument_values.json"  # optional

# If you have an Excel file, uncomment and run:
# parse_telecom_functions("path/to/telecom_functions.xlsx", FUNCTION_LIBRARY_PATH)

# Training configuration (shared across algorithms)
TRAIN_CONFIG = {
    "model": {
        "name": "unsloth/Qwen3-4B-Base",
        "max_seq_length": 2048,
        "load_in_4bit": True,
        "fast_inference": True,
        "gpu_memory_utilization": 0.7,
    },
    "lora": {
        "r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "bias": "none",
        "use_gradient_checkpointing": "unsloth",
    },
    "training": {
        "output_dir": f"outputs/{ALGORITHM}_model",
        "learning_rate": 5e-6,
        "adam_beta1": 0.9,
        "adam_beta2": 0.99,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "num_generations": 8,
        "max_steps": 200,          # reduce for quick test; increase for full run
        "save_steps": 50,
        "logging_steps": 1,
        "max_grad_norm": 0.1,
        "report_to": "none",
        "seed": 3407,
    },
    "data": {
        "train_path": "data/processed/raw_train_dataset.jsonl",   # generated by dataset_generator
        "test_path": "data/processed/raw_test_dataset.jsonl",
        "max_prompt_length": 1024,
        "max_completion_length": 512,
    },
    "grpo": {
        "temperature": 1.0,
        "epsilon": 0.2,
        "epsilon_high": 0.28,
        "loss_type": "grpo",
        "mask_truncated_completions": True,
    },
    # Algorithm-specific configs
    "gvpo": {
        "shaping_coeff": 1.0,
        "args_threshold": 0.8,
    },
    "rc_grpo": {
        "high_fraction": 0.5,   # will be overridden if RCTP dataset provided
    },
    "autotool": {
        "entropy_coeff_reasoning": 0.01,
        "entropy_coeff_tool": 0.005,
        "entropy_coeff_other": 0.001,
    },
    "awpo": {
        "clip_shrink_coeff": 0.2,
        "outcome_weights": {"func_selection": 0.4, "args_accuracy": 0.3, "execution": 0.3},
    },
}

In [ ]:
# Load function library and argument values (if available)
function_library = load_function_library(FUNCTION_LIBRARY_PATH)
print(f"Loaded {len(function_library)} functions")

argument_values = None
if ARGUMENT_VALUES_PATH.exists():
    with open(ARGUMENT_VALUES_PATH, "r") as f:
        argument_values = json.load(f)
    print(f"Loaded argument values catalog with {len(argument_values)} parameters")
else:
    print("No argument_values.json; will skip argument values in prompts.")

# If dataset not exists, you can generate using dataset_generator.py
# But that requires API key. We'll assume you have pre-generated dataset.
# If not, you can use a small synthetic dataset for testing.

In [ ]:
# Load model and tokenizer
model, tokenizer = load_model(TRAIN_CONFIG)
print("Model and tokenizer loaded.")

# Register special tokens if using RC-GRPO
if ALGORITHM == "rc_grpo":
    RCGRPOTrainer.register_reward_tokens(tokenizer)
    model.resize_token_embeddings(len(tokenizer))

# Load dataset
dataset = load_grpo_dataset(
    TRAIN_CONFIG["data"]["train_path"],
    function_library,
    tokenizer,
    argument_values_catalog=argument_values,
    # telco_retriever can be passed if you want live retrieval
)

# Build GRPOConfig
grpo_args = build_grpo_config(TRAIN_CONFIG, output_dir=TRAIN_CONFIG["training"]["output_dir"])

# Instantiate trainer based on algorithm
if ALGORITHM == "gvpo":
    trainer = GVPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[gvpo_reward_func, format_reward],
        args=grpo_args,
        train_dataset=dataset,
        gvpo_config=TRAIN_CONFIG.get("gvpo", {}),
    )
elif ALGORITHM == "rc_grpo":
    # Inject diverse reward tokens if RC-GRPO
    # For Phase 2, we need to set high_fraction; here we use default 0.5
    high_fraction = TRAIN_CONFIG.get("rc_grpo", {}).get("high_fraction", 0.5)
    dataset = inject_diverse_reward_tokens(
        dataset,
        num_generations=TRAIN_CONFIG["training"]["num_generations"],
        high_fraction=high_fraction,
    )
    trainer = RCGRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[rc_grpo_reward_func, rc_grpo_format_func],
        args=grpo_args,
        train_dataset=dataset,
    )
elif ALGORITHM == "autotool":
    # Register special tokens for AutoTool
    special = ["<reasoning>", "</reasoning>", "<tool_call>", "</tool_call>"]
    tokenizer.add_special_tokens({"additional_special_tokens": special})
    model.resize_token_embeddings(len(tokenizer))
    trainer = AutoToolTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[autotool_reward_func, format_reward],
        args=grpo_args,
        train_dataset=dataset,
        autotool_config=TRAIN_CONFIG.get("autotool", {}),
    )
elif ALGORITHM == "awpo":
    trainer = AWPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[awpo_reward_func, format_reward],
        args=grpo_args,
        train_dataset=dataset,
        awpo_config=TRAIN_CONFIG.get("awpo", {}),
    )
else:  # vanilla GRPO
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[gvpo_reward_func, format_reward],  # use gvpo_reward_func as outcome only
        args=grpo_args,
        train_dataset=dataset,
    )

print(f"Trainer initialized for {ALGORITHM}")

In [ ]:
# Start training
print("Starting training...")
trainer.train()
print("Training completed.")

In [ ]:
# Save model and tokenizer
output_dir = grpo_args.output_dir
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model saved to {output_dir}")

In [ ]:
# Load test dataset
test_dataset_path = TRAIN_CONFIG["data"]["test_path"]
if Path(test_dataset_path).exists():
    # Initialize retriever and sandbox
    retriever = FunctionRetriever(function_library, method="hybrid")
    sandbox = Sandbox(function_library)

    eval_result = evaluate_model(
        model_path=output_dir,
        test_dataset_path=test_dataset_path,
        function_library=function_library,
        retriever=retriever,
        sandbox=sandbox,
        top_k=5,
        max_new_tokens=512,
        model_name_tag=ALGORITHM,
        argument_values=argument_values,
    )
    print("Evaluation result:", eval_result["aggregate"])

    # Generate report (if multiple models evaluated, collect results)
    generate_report([eval_result])
else:
    print("Test dataset not found; skipping evaluation.")

### Unsloth

Goal: To convert `Qwen3-4B-Base` into a reasoning model via GRPO by using OpenR1's Math dataset.

We first pre fine-tune the model to make GRPO skip trying to match formatting - this speeds GRPO up.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Base",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

### GRPO chat template
Since we're using a base model, we should set a chat template. You can make your own chat template as well!
1. DeepSeek uses `<think>` and `</think>`, but this is **not** necessary - you can customize it however you like!
2. A `system_prompt` is recommended to at least guide the model's responses.

In [ ]:
reasoning_start = "<start_working_out>" # Acts as think-open tag
reasoning_end   = "<end_working_out>"   # Acts as think-close tag
solution_start  = "<SOLUTION>"
solution_end    = "</SOLUTION>"

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
Place it between {reasoning_start} and {reasoning_end}.
Then, provide your solution between {solution_start}{solution_end}"""
system_prompt

We create a simple chat template below. Notice `add_generation_prompt` includes prepending `<start_working_out>` to guide the model to start its reasoning process.

In [ ]:
chat_template = \
    "{% if messages[0]['role'] == 'system' %}"\
        "{{ messages[0]['content'] + eos_token }}"\
        "{% set loop_messages = messages[1:] %}"\
    "{% else %}"\
        "{{ '{system_prompt}' + eos_token }}"\
        "{% set loop_messages = messages %}"\
    "{% endif %}"\
    "{% for message in loop_messages %}"\
        "{% if message['role'] == 'user' %}"\
            "{{ message['content'] }}"\
        "{% elif message['role'] == 'assistant' %}"\
            "{{ message['content'] + eos_token }}"\
        "{% endif %}"\
    "{% endfor %}"\
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"\
    "{% endif %}"

# Replace with our specific template:
chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{system_prompt}'")\
    .replace("'{reasoning_start}'", f"'{reasoning_start}'")
tokenizer.chat_template = chat_template

Let's see how our chat template behaves on an example:

In [ ]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "What is 1+1?"},
    {"role" : "assistant", "content" : f"{reasoning_start}I think it's 2.{reasoning_end}{solution_start}2{solution_end}"},
    {"role" : "user", "content" : "What is 2+2?"},
], tokenize = False, add_generation_prompt = True)

### Pre fine-tuning for formatting
We now use a subset of NVIDIA's [Open Math Reasoning dataset](https://huggingface.co/datasets/nvidia/OpenMathReasoning) which was filtered to only include high quality DeepSeek R1 traces.

We'll only filter ~59 or so examples to first "prime" / pre fine-tune the model to understand our custom GRPO formatting.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

We have to format the dataset to follow our GRPO style formatting:

In [ ]:
def format_dataset(x):
    expected_answer = x["expected_answer"]
    problem = x["problem"]

    # Remove generated think tags
    thoughts = x["generated_solution"]
    thoughts = thoughts.replace("<think>", "").replace("</think>", "")

    # Strip newlines on left and right
    thoughts = thoughts.strip()
    # Add our custom formatting
    final_prompt = \
        reasoning_start + thoughts + reasoning_end + \
        solution_start + expected_answer + solution_end
    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : final_prompt},
    ]

dataset["Messages"] = dataset.apply(format_dataset, axis = 1)

Check to see if it worked:

In [ ]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

Let's truncate the pre fine-tuning dataset to `max_seq_length/2` since we don't want too long reasoning traces.

Note this might take 2 minutes!

In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length/2].copy()
dataset.shape

We then tokenize the messages and convert it to a Hugging Face compatible dataset format:

In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

Let's now pre fine-tune the model so it follows our custom GRPO formatting!

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 1, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 2, # Set this for 1 full training run.
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer.train()

Let's check if the model has learnt to follow the custom format:

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

Yes it did follow the formatting! Great! Let's remove some items before the GRPO step

In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

### Data Prep
<a name="Data"></a>

We're using Hugging Face's [Open R1 Math dataset](https://huggingface.co/datasets/open-r1/DAPO-Math-17k-Processed). You can also utilize OpenAI's famous [GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

Let's look at the first row:

In [ ]:
dataset[0]["prompt"]

In [ ]:
dataset[0]["solution"]

In GSM8K, we notice all answers like about have a ####, so we extract it. But for the Open R1 dataset, we can skip the below.

In [ ]:
def extract_hash_answer(text):
    # if "####" not in text: return None
    # return text.split("####")[1].strip()
    return text
extract_hash_answer(dataset[0]["solution"])

Let's map the dataset! and see the first row:

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

We create a regex format to match the reasoning sections and answers:

In [ ]:
import re

# Add optional EOS token matching
solution_end_regex = r"</SOLUTION>[\s]{0,}" + \
    "(?:" + re.escape(tokenizer.eos_token) + ")?"

match_format = re.compile(
    rf"{reasoning_end}.*?"\
    rf"{solution_start}(.+?){solution_end_regex}"\
    rf"[\s]{{0,}}$",
    flags = re.MULTILINE | re.DOTALL
)
match_format

We verify it works:

In [ ]:
match_format.findall(
    "Let me think!<end_working_out>"\
    f"<SOLUTION>\n2\n</SOLUTION>",
)

In [ ]:
match_format.findall(
    "<start_working_out>Let me think!<end_working_out>"\
    f"<SOLUTION>  2  </SOLUTION>\n\n",
)

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward the opening tag since we always prepend it!
        # score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        score += 0.5 if response.count(solution_start)  == 1 else -1.0
        score += 0.5 if response.count(solution_end)    == 1 else -1.0
        scores.append(score)
    return scores

Finally, we want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.

We also remove possible commas for example as in 123,456

In [ ]:
match_numbers = re.compile(
    solution_start + r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("<SOLUTION>  0.34  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  123,456  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>  -0.234  </SOLUTION>"))
print(match_numbers.findall("<SOLUTION>17</SOLUTION>"))

We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_saved_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test:

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "What is the sqrt of 101?"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("qwen_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("qwen_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/qwen_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("qwen_lora")
    tokenizer.save_pretrained("qwen_lora")
if False:
    model.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/qwen_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

Now, use the `qwen_finetune.Q8_0.gguf` file or `qwen_finetune.Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).